In [ ]:
import subprocess
import sys

# Install packages directly into the active Python kernel
# %%bash cells install into a subprocess shell that doesn't persist to the kernel
packages = ['numpy', 'scipy', 'matplotlib', 'pandas', 'readgssi']
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"FAILED: {pkg}\n{result.stderr}")
    else:
        print(f"OK: {pkg}")

# Torch is pre-installed on Kaggle GPU
import torch
print(f"OK: torch {torch.__version__}")

import readgssi
print(f"OK: readgssi {readgssi.__version__}")

In [ ]:
import os
import numpy as np
from scipy.signal import hilbert as sp_hilbert
from scipy.signal import resample as scipy_resample

# Universal model constants
MAX_DEPTH_INCHES  = 10.0
MAX_DEPTH_CM      = MAX_DEPTH_INCHES * 2.54   # 25.4 cm
TARGET_SAMPLES    = 512
SEARCH_START      = 30    # wider window — deeper rebar needs earlier start
SEARCH_END        = 400   # covers up to 10 inches
EPSR_DEFAULT      = 9.0   # fallback only when surface reflection unavailable
EPSR_MIN          = 4.0   # physical minimum for concrete
EPSR_MAX          = 16.0  # physical maximum for saturated concrete

# Proceq RIS Hi-Bright (from RIS_Hi-Bright.xml)
RANGE_NS_PROCEQ   = 16.0
SURFACE_SEARCH    = 20    # samples to search for surface reflection

# GSSI SIR 30
RANGE_NS_GSSI     = 15.0

# Local test mode: auto-detects when running on the original Mac
LOCAL_TEST = os.path.exists('/Users/aidenerard/Desktop/verus-workspace/verus/data/B440029/')

if LOCAL_TEST:
    PROCEQ_DATA_DIR = '/Users/aidenerard/Desktop/verus-workspace/verus/data/CJ245308_241003_001/Data/'
    B440029_CSV     = '/Users/aidenerard/Desktop/verus-workspace/verus/data/B440029/B440029 Rebar Depth Report (1).csv'
    B170020_CSV     = '/Users/aidenerard/Desktop/verus-workspace/verus/data/B170020/B170020 Rebar Depth Report (1).csv'
    DZT_B440        = '/Users/aidenerard/Desktop/verus-workspace/verus/data/B440029/raw data/'
    DZT_B170        = '/Users/aidenerard/Desktop/verus-workspace/verus/data/B170020/raw data/'
    WORKING_DIR     = '/Users/aidenerard/Desktop/verus-workspace/verus/output/models/'
    os.makedirs(WORKING_DIR, exist_ok=True)
    print("LOCAL TEST MODE — using Mac paths")
else:
    PROCEQ_DATA_DIR = '/kaggle/input/datasets/aidenerard/terracon-proceq-bridge/CJ245308_241003_001/Data/'
    B440029_CSV     = '/kaggle/input/datasets/aidenerard/infrasense-depth-labels/B440029 Rebar Depth Report (1).csv'
    B170020_CSV     = '/kaggle/input/datasets/aidenerard/infrasense-depth-labels/B170020 Rebar Depth Report (1).csv'
    DZT_B440        = '/kaggle/input/datasets/aidenerard/infrasense-raw-data/B440029/raw data/'
    DZT_B170        = '/kaggle/input/datasets/aidenerard/infrasense-raw-data/B170020/raw data/'
    WORKING_DIR     = '/kaggle/working/'
    print("KAGGLE MODE — using Kaggle dataset paths")

In [ ]:
def calculate_epsr_from_amplitudes(amp_surface, amp_plate):
    """
    Surface reflection method for per-point dielectric calculation.
    Identical to Infrasense's L2Dielectric column per Ken's email.

    Physics: At an air-concrete interface, the reflection coefficient
    depends on the dielectric contrast. Using Fresnel equation for
    normal incidence:
        r = (1 - sqrt(epsr)) / (1 + sqrt(epsr))
        amp_surface / amp_plate = |r|
    Solving for epsr:
        sqrt(epsr) = (1 + r) / (1 - r)
        epsr = ((1 + r) / (1 - r))^2
    """
    if amp_plate <= 0 or amp_surface <= 0:
        return EPSR_DEFAULT
    r = np.clip(amp_surface / amp_plate, 0.01, 0.99)
    epsr = ((1 + r) / (1 - r)) ** 2
    return float(np.clip(epsr, EPSR_MIN, EPSR_MAX))


def calculate_epsr_from_trace(trace, plate_amp, range_ns):
    """
    Estimate per-trace dielectric from surface reflection in raw trace.
    Finds peak amplitude in first SURFACE_SEARCH samples (surface wavelet).
    plate_amp: reference amplitude from metal plate calibration sweep.
    """
    n_samples    = len(trace)
    search_end   = min(SURFACE_SEARCH, n_samples // 4)
    surface_amp  = float(np.abs(trace[:search_end]).max())
    return calculate_epsr_from_amplitudes(surface_amp, plate_amp)


def depth_inches_from_pick(pick_sample, n_samples, epsr, range_ns):
    """
    Convert sample index to depth in inches using local dielectric.
    This is the core depth calculation — epsr varies per point.
    """
    ns_per_sample = range_ns / n_samples
    velocity      = 0.15 / np.sqrt(epsr)   # m/ns
    depth_m       = (pick_sample * ns_per_sample * velocity) / 2.0
    return depth_m * 39.3701               # inches


def normalize_depth(depth_inches):
    return float(np.clip(depth_inches / MAX_DEPTH_INCHES, 0.0, 1.0))


def denormalize_depth(norm):
    return float(norm * MAX_DEPTH_INCHES)


def preprocess_trace(raw_trace, target_samples=TARGET_SAMPLES):
    """
    Standardize any GPR trace to target_samples, normalized.
    Format-agnostic — works for GSSI, Proceq, any system.
    NOTE: normalization is applied AFTER dielectric/depth calculation.
    """
    if len(raw_trace) != target_samples:
        raw_trace = scipy_resample(raw_trace, target_samples)
    raw_trace = raw_trace - raw_trace.mean()
    max_abs   = np.abs(raw_trace).max()
    if max_abs > 0:
        raw_trace = raw_trace / max_abs
    return raw_trace.astype(np.float32)


In [ ]:
_SCAN_MAGIC    = b'VH01SW'
_D_BLOCK_SIZE  = 0x040C
_D_HEADER_SIZE = 16
_D_N_SAMPLES   = 510
_D_N_REF       = 16       # first 16 D-blocks are reference sweeps
_D_MARKER      = b'D' + bytes(1)  # b'D\x00' D-block start tag


def read_scan_raw(scan_path):
    """
    Read raw D-block traces from Proceq PRC scan file.
    Returns (ref_traces, data_traces) — ref sweeps used for plate_amp.
    """
    with open(scan_path, 'rb') as f:
        raw = f.read()
    if raw[:6] != _SCAN_MAGIC:
        return None, None
    offset  = 0x027C
    n_total = (len(raw) - offset) // _D_BLOCK_SIZE
    ref_traces  = []
    data_traces = []
    for i in range(n_total):
        bs = offset + i * _D_BLOCK_SIZE
        if raw[bs:bs+2] != _D_MARKER:
            continue
        payload = raw[bs+_D_HEADER_SIZE : bs+_D_HEADER_SIZE+_D_N_SAMPLES*2]
        if len(payload) < _D_N_SAMPLES * 2:
            continue
        samples = np.frombuffer(payload, dtype=np.int16).astype(np.float32)
        if i < _D_N_REF:
            ref_traces.append(samples)
        else:
            data_traces.append(samples)
    return (np.array(ref_traces)  if ref_traces  else None,
            np.array(data_traces) if data_traces else None)


def get_plate_amp_proceq(ref_traces):
    """
    Compute plate reference amplitude from Proceq reference sweeps.
    Reference sweeps (first 16 D-blocks) are acquired over a metal plate
    during system calibration — equivalent to Infrasense's ampPlate column.
    """
    if ref_traces is None or len(ref_traces) == 0:
        return None
    surface_amps = np.abs(ref_traces[:, :SURFACE_SEARCH]).max(axis=1)
    return float(surface_amps.mean())


def load_proceq_dataset():
    """
    Load Terracon Proceq data with per-trace dielectric correction.
    Labels: Hilbert picks converted to depth using local epsr per trace.
    """
    import glob
    scan_files = sorted(glob.glob(PROCEQ_DATA_DIR + 'PRC_*.scan'))
    odd_scans  = [f for f in scan_files
                  if int(f.split('PRC_')[1].replace('.scan', '')) % 2 == 1]

    CHANNELS_PER_SWATH = 4
    swath_groups = [odd_scans[i:i + CHANNELS_PER_SWATH]
                    for i in range(0, len(odd_scans), CHANNELS_PER_SWATH)]

    swath_traces, swath_labels, swath_ids = [], [], []

    for swath_idx, swath_scans in enumerate(swath_groups):
        t_swath, l_swath = [], []
        for scan_path in swath_scans:
            ref_traces, data_traces = read_scan_raw(scan_path)
            if data_traces is None or len(data_traces) == 0:
                continue
            plate_amp = get_plate_amp_proceq(ref_traces)

            for raw in data_traces:
                if plate_amp is not None:
                    epsr = calculate_epsr_from_trace(raw, plate_amp, RANGE_NS_PROCEQ)
                else:
                    epsr = EPSR_DEFAULT

                envelope = np.abs(sp_hilbert(raw))
                s_start  = int(SEARCH_START * _D_N_SAMPLES / TARGET_SAMPLES)
                s_end    = int(SEARCH_END   * _D_N_SAMPLES / TARGET_SAMPLES)
                s_end    = min(s_end, _D_N_SAMPLES - 1)
                pick     = int(np.argmax(envelope[s_start:s_end])) + s_start

                depth_in = depth_inches_from_pick(pick, _D_N_SAMPLES, epsr, RANGE_NS_PROCEQ)
                if depth_in < 0.3 or depth_in > MAX_DEPTH_INCHES:
                    continue

                proc  = preprocess_trace(raw, TARGET_SAMPLES)
                label = normalize_depth(depth_in)
                t_swath.append(proc)
                l_swath.append(label)

        if not t_swath:
            print(f"  Swath {swath_idx+1}: no valid traces")
            continue

        t_arr = np.array(t_swath, dtype=np.float32)
        l_arr = np.array(l_swath, dtype=np.float32)
        swath_traces.append(t_arr)
        swath_labels.append(l_arr)
        swath_ids.append(swath_idx)

        depths = l_arr * MAX_DEPTH_INCHES
        print(f"  Swath {swath_idx+1}: {len(t_arr):,} traces  "
              f"depth {depths.mean():.2f}\" mean  "
              f"[{depths.min():.2f}\"–{depths.max():.2f}\"]")

    print(f"\nProceq total: {sum(len(t) for t in swath_traces):,} traces")
    return swath_traces, swath_labels, swath_ids, len(swath_traces)


In [ ]:
# Ensure readgssi is available
try:
    import readgssi
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'readgssi', '-q'], check=True)
    import readgssi

import pandas as pd
import numpy as np
import os

print(f"readgssi version: {readgssi.__version__}")

inf_traces, inf_labels, inf_ids = [], [], []

for csv_path, dzt_dir, bridge_id in [
    (B440029_CSV, DZT_B440, 'B440029'),
    (B170020_CSV, DZT_B170, 'B170020'),
]:
    print(f"\n{'='*50}")
    print(f"Loading {bridge_id}")
    print(f"  CSV: {csv_path}")
    print(f"  DZT dir: {dzt_dir}")

    # Check CSV exists
    if not os.path.exists(csv_path):
        print(f"  ERROR: CSV not found")
        continue

    # Load CSV
    df = pd.read_csv(csv_path)
    print(f"  CSV rows raw: {len(df)}")
    df = df.dropna(subset=['L2Depth_inches', 'L2Dielectric', 'ampSurface', 'ampPlate'])
    df = df[(df['L2Depth_inches'] > 0.3) & (df['L2Depth_inches'] <= MAX_DEPTH_INCHES)]
    print(f"  CSV rows after filter: {len(df)}")

    # Check DZT dir exists
    if not os.path.exists(dzt_dir):
        print(f"  ERROR: DZT directory not found")
        continue

    dzt_files_available = os.listdir(dzt_dir)
    print(f"  DZT files available: {dzt_files_available[:5]}")

    loaded_traces = 0
    skipped_files = 0

    for fname, group in df.groupby('preProcessedFileName'):
        raw_fname = (fname
                     .replace('_PREP', '')
                     .replace('_ch01', '')
                     .replace('_ch02', '')
                     .replace('_r', ''))
        dzt_path = os.path.join(dzt_dir, raw_fname)

        if not os.path.exists(dzt_path):
            print(f"  MISSING: {raw_fname}")
            skipped_files += 1
            continue

        print(f"  Loading: {raw_fname} ({len(group)} labels)")

        try:
            header, arrays, _ = readgssi.readgssi(
                infile=dzt_path, zero=[0], verbose=False)

            if not arrays or len(arrays) == 0:
                print(f"    ERROR: no arrays returned")
                continue

            traces = arrays[0].T.astype(np.float32)
            n_traces, n_samp = traces.shape
            print(f"    Traces shape: {traces.shape}")

            file_traces = 0
            for _, row in group.iterrows():
                scan_idx = int(row['scanNumber'])
                if scan_idx >= n_traces:
                    continue
                raw_trace = traces[scan_idx]
                label = normalize_depth(row['L2Depth_inches'])
                proc = preprocess_trace(raw_trace, TARGET_SAMPLES)
                inf_traces.append(proc)
                inf_labels.append(label)
                inf_ids.append(f"{bridge_id}_{fname}")
                file_traces += 1

            print(f"    Loaded {file_traces} traces from this file")
            loaded_traces += file_traces

        except Exception as e:
            import traceback
            print(f"    EXCEPTION: {e}")
            traceback.print_exc()
            continue

    print(f"\n{bridge_id} summary:")
    print(f"  Loaded: {loaded_traces} traces")
    print(f"  Skipped files: {skipped_files}")

print(f"\n{'='*50}")
print(f"INFRASENSE TOTAL: {len(inf_traces)} traces")
if len(inf_traces) > 0:
    depths = [l * MAX_DEPTH_INCHES for l in inf_labels]
    print(f"Depth range: {min(depths):.2f}\" – {max(depths):.2f}\"")

In [ ]:
print("=" * 60)
print("Loading Proceq dataset (Hilbert + per-trace dielectric)...")
print("=" * 60)
proc_traces, proc_labels, proc_ids, n_swaths = load_proceq_dataset()

print("\n" + "=" * 60)
print("Loading Infrasense dataset (ground truth L2 depths)...")
print("=" * 60)
inf_traces, inf_labels, inf_ids = load_infrasense_dataset()

# Concatenate Proceq (with graceful guard in case dir is missing)
if proc_traces:
    proc_t = np.concatenate(proc_traces)
    proc_l = np.concatenate(proc_labels)
else:
    proc_t = np.empty((0, TARGET_SAMPLES), dtype=np.float32)
    proc_l = np.empty(0, dtype=np.float32)
    print("WARNING: No Proceq traces — check PROCEQ_DATA_DIR")

# Infrasense — weight 3× (smaller but higher quality ground truth)
if inf_traces:
    inf_t = np.tile(np.array(inf_traces, dtype=np.float32), (3, 1))
    inf_l = np.tile(np.array(inf_labels, dtype=np.float32),  3)
else:
    inf_t = np.empty((0, TARGET_SAMPLES), dtype=np.float32)
    inf_l = np.empty(0, dtype=np.float32)
    print("WARNING: No Infrasense traces — training on Proceq only")

all_t = np.concatenate([proc_t, inf_t])
all_l = np.concatenate([proc_l, inf_l])

print(f"\nCombined dataset:")
print(f"  Proceq:     {len(proc_t):,} traces")
print(f"  Infrasense: {len(inf_t)//3 if len(inf_t) > 0 else 0:,} × 3 weight")
print(f"  Total:      {len(all_t):,}")
print(f"  Depth range: {all_l.min()*MAX_DEPTH_INCHES:.2f}\" – "
      f"{all_l.max()*MAX_DEPTH_INCHES:.2f}\"")

# Train/val split — by bridge for true generalization test
# Train: Proceq swaths 1-10 + B440029
# Val:   Proceq swaths 11-14 + B170020
if proc_traces:
    proc_train_t = np.concatenate(proc_traces[:10]) if proc_traces[:10] else np.empty((0, TARGET_SAMPLES), dtype=np.float32)
    proc_train_l = np.concatenate(proc_labels[:10]) if proc_labels[:10] else np.empty(0, dtype=np.float32)
    proc_val_t   = np.concatenate(proc_traces[10:]) if proc_traces[10:] else np.empty((0, TARGET_SAMPLES), dtype=np.float32)
    proc_val_l   = np.concatenate(proc_labels[10:]) if proc_labels[10:] else np.empty(0, dtype=np.float32)
else:
    proc_train_t = proc_val_t = np.empty((0, TARGET_SAMPLES), dtype=np.float32)
    proc_train_l = proc_val_l = np.empty(0, dtype=np.float32)

inf_b440_mask = [i for i, id_ in enumerate(inf_ids) if 'B440029' in id_]
inf_b170_mask = [i for i, id_ in enumerate(inf_ids) if 'B170020' in id_]

if inf_traces:
    inf_arr_t = np.array(inf_traces, dtype=np.float32)
    inf_arr_l = np.array(inf_labels, dtype=np.float32)
    if inf_b440_mask:
        inf_train_t = np.tile(inf_arr_t[inf_b440_mask], (3, 1))
        inf_train_l = np.tile(inf_arr_l[inf_b440_mask],  3)
    else:
        inf_train_t = np.empty((0, TARGET_SAMPLES), dtype=np.float32)
        inf_train_l = np.empty(0, dtype=np.float32)
    if inf_b170_mask:
        inf_val_t = inf_arr_t[inf_b170_mask]
        inf_val_l = inf_arr_l[inf_b170_mask]
    else:
        inf_val_t = np.empty((0, TARGET_SAMPLES), dtype=np.float32)
        inf_val_l = np.empty(0, dtype=np.float32)
else:
    inf_train_t = inf_val_t = np.empty((0, TARGET_SAMPLES), dtype=np.float32)
    inf_train_l = inf_val_l = np.empty(0, dtype=np.float32)

X_train = np.concatenate([proc_train_t, inf_train_t])
y_train = np.concatenate([proc_train_l, inf_train_l])
X_val   = np.concatenate([proc_val_t,   inf_val_t])
y_val   = np.concatenate([proc_val_l,   inf_val_l])

print(f"\nTrain: {len(X_train):,} traces")
print(f"Val:   {len(X_val):,}   traces")
print(f"  Proceq val: {len(proc_val_t):,}  Infrasense val (B170020): {len(inf_val_t):,}")


In [ ]:
import torch
import torch.nn as nn


class TemporalAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.score = nn.Linear(channels, 1)

    def forward(self, x):
        w = torch.softmax(self.score(x.transpose(1, 2)), dim=1)
        return (x.transpose(1, 2) * w).sum(dim=1)


class HorizonCNN(nn.Module):
    """Universal rebar depth regression. Input (batch,1,512) → (batch,) 0-1."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1,   32,  7, padding=3), nn.BatchNorm1d(32),  nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32,  64,  5, padding=2), nn.BatchNorm1d(64),  nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64,  128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.MaxPool1d(2),
        )
        self.attn = TemporalAttention(128)
        self.head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.head(self.attn(self.conv(x))).squeeze(-1)


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model     = HorizonCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)
criterion = nn.SmoothL1Loss()

train_ds = TensorDataset(torch.tensor(X_train).unsqueeze(1), torch.tensor(y_train))
val_ds   = TensorDataset(torch.tensor(X_val).unsqueeze(1),   torch.tensor(y_val))

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True,
                          pin_memory=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False,
                          pin_memory=True)

EPOCHS, PATIENCE = 100, 20
best_mae, patience_count = float('inf'), 0

print(f"\n{'Ep':>4} {'TR_loss':>9} {'Val_loss':>9} "
      f"{'MAE_cm':>8} {'RMSE_cm':>9} "
      f"{'MAE_in':>7} {'Best_MAE':>9} {'LR':>10}")
print("-" * 80)

for epoch in range(1, EPOCHS + 1):
    # Training
    model.train()
    tr_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        if torch.rand(1) > 0.5:
            xb = xb + torch.randn_like(xb) * 0.01
        if torch.rand(1) > 0.5:
            xb = xb * (0.9 + torch.rand(1).item() * 0.2)
        pred = model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        tr_loss += loss.item()

    # Validation
    model.eval()
    preds, targets = [], []
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            val_loss += criterion(pred, yb).item()
            preds.extend(pred.cpu().numpy())
            targets.extend(yb.cpu().numpy())

    preds   = np.array(preds)   * MAX_DEPTH_INCHES
    targets = np.array(targets) * MAX_DEPTH_INCHES
    mae_in  = np.abs(preds - targets).mean()
    mae_cm  = mae_in * 2.54
    rmse_cm = np.sqrt(((preds * 2.54 - targets * 2.54) ** 2).mean())

    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    if mae_cm < best_mae:
        best_mae = mae_cm
        patience_count = 0
        torch.save(model.state_dict(),
                   os.path.join(WORKING_DIR, 'horizon_model_best.pth'))
    else:
        patience_count += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>4} {tr_loss/len(train_loader):>9.5f} "
              f"{val_loss/len(val_loader):>9.5f} "
              f"{mae_cm:>8.3f} {rmse_cm:>9.3f} "
              f"{mae_in:>7.3f} {best_mae:>9.3f} {lr:>10.2e}")

    if patience_count >= PATIENCE:
        print(f"Early stop at epoch {epoch}")
        break


In [ ]:
# Load best model and report separately for each dataset
model.load_state_dict(torch.load(
    os.path.join(WORKING_DIR, 'horizon_model_best.pth'),
    map_location=device,
))
model.eval()


def eval_dataset(traces, labels, name):
    if len(traces) == 0:
        print(f"{name}: no data")
        return
    ds = TensorDataset(torch.tensor(traces).unsqueeze(1), torch.tensor(labels))
    dl = DataLoader(ds, batch_size=512, shuffle=False)
    preds, tgts = [], []
    with torch.no_grad():
        for xb, yb in dl:
            preds.extend(model(xb.to(device)).cpu().numpy())
            tgts.extend(yb.numpy())
    p = np.array(preds) * MAX_DEPTH_INCHES
    t = np.array(tgts)  * MAX_DEPTH_INCHES
    mae_in  = np.abs(p - t).mean()
    rmse_in = np.sqrt(((p - t) ** 2).mean())
    print(f"{name}:")
    print(f"  MAE:  {mae_in * 2.54:.3f} cm  ({mae_in:.3f} inches)")
    print(f"  RMSE: {rmse_in * 2.54:.3f} cm  ({rmse_in:.3f} inches)")
    print(f"  Depth range pred:   {p.min():.2f}\" – {p.max():.2f}\"")
    print(f"  Depth range actual: {t.min():.2f}\" – {t.max():.2f}\"")


print("\n=== Validation Results by Dataset ===")
eval_dataset(proc_val_t,  proc_val_l,  "Proceq val (swaths 11-14)")
if len(inf_val_t) > 0:
    eval_dataset(inf_val_t, inf_val_l, "Infrasense val (B170020)")

# Save final model
import shutil
shutil.copy(os.path.join(WORKING_DIR, 'horizon_model_best.pth'),
            os.path.join(WORKING_DIR, 'horizon_model_universal.pth'))
print(f"\nSaved: {WORKING_DIR}horizon_model_universal.pth")
print(f"MAX_DEPTH_INCHES = {MAX_DEPTH_INCHES}")
print(f"Update gpr_ensemble.py: MAX_REBAR_DEPTH_CM = {MAX_DEPTH_INCHES * 2.54}")
